# Clinical Problem and Objective

**The Clinical Problem:** Post-myocardial infarction management requires predicting severe complications (e.g., arrhythmias, heart failure, ruptures) rather than simply binary survival. These complications are rare, resulting in extreme class imbalance. This frequently causes traditional machine learning models to default to predicting the majority class ("no complication") to artificially inflate accuracy
a phenomenon known as the Accuracy Paradox.

**Project Objective:** 
* Build a high-sensitivity predictive system optimized for **Recall**.
* Accurately forecast 12 distinct clinical targets simultaneously.
* Handle a "small data" regime without severe overfitting.

In [1]:
import warnings
warnings.filterwarnings('ignore')
!pip install ucimlrepo xgboost autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 8.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.6/227.6 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.9/98.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.4/74.4 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 38.1 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
datasets 4.8.3 requires pyarrow>=21.0.0, but you have pyarrow 20.0

# Dataset Overview

**Source:** *Myocardial Infarction Complications Data Set* (UCI Machine Learning Repository).

**Data Dimensions:**
* **Instances:** 1,700 patients.
* **Input Variables (Features):** 111 (Demographics, Anamnesis, ECG findings, Biomarkers, Vitals).
* **Targets (12 Outcomes):** 11 Binary Complications and 1 Multiclass Target (Lethal Outcome).

**Sparsity and Missing Data:**
The dataset exhibits severe Missing Not At Random (MNAR) patterns, representing real-world clinical triage. Extreme variables reach >60% missingness (e.g., Family history, Systolic BP in the ER). Aggressive imputation can destroy the predictive signal inherent in the "absence" of a clinical test.

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, f1_score, confusion_matrix
import xgboost as xgb
from ucimlrepo import fetch_ucirepo
from autogluon.tabular import TabularDataset, TabularPredictor

# 1. Fetch data from the UCI repository
print("Fetching dataset...")
mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()
variables_info = mi_data.variables

# 2. Identify Column Types
cat_cols_info = variables_info[(variables_info['role'] == 'Feature') & (variables_info['type'] == 'Categorical')]['name'].tolist()
categorical_cols = [c for c in cat_cols_info if c in X_full.columns]
numeric_cols = [c for c in X_full.columns if c not in categorical_cols]

# 3. Fill missing targets with the most frequent value (mode)
y_full = y_full.fillna(y_full.mode().iloc[0])

target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]
multiclass_target = target_names[-1]

print(f"Loaded structure: {X_full.shape[0]} patients, {X_full.shape[1]} features, {y_full.shape[1]} targets.")

Fetching dataset...
Loaded structure: 1700 patients, 111 features, 12 targets.


# The Temporal Challenge and Data Masking

Clinical information is unlocked chronologically over 72 hours. A static model risks **Data Leakage** (using future treatments to predict present risks).

**Masking Strategy:**
We create distinct datasets based on the clinical timeline. When predicting at *Admission*, variables from Days 1, 2, and 3 are strictly masked to ensure valid real-time simulations. The Temporal Augmentation function concatenates these stages to maximize the sample size.

In [3]:
def generate_temporal_datasets(X_data):
    """
    Generates four distinct datasets based on the clinical timeline.
    Removes future columns to prevent data leakage.
    """
    day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
    day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
    day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']

    datasets = {}
    drop_admission = day_1_cols + day_2_cols + day_3_cols
    datasets['admission'] = X_data.drop(columns=[c for c in drop_admission if c in X_data.columns])

    drop_day_1 = day_2_cols + day_3_cols
    datasets['day_1'] = X_data.drop(columns=[c for c in drop_day_1 if c in X_data.columns])

    drop_day_2 = day_3_cols
    datasets['day_2'] = X_data.drop(columns=[c for c in drop_day_2 if c in X_data.columns])

    datasets['day_3'] = X_data.copy()
    return datasets

def create_augmented_dataset(X_base, y_base):
    """
    Concatenates the 4 timelines into a single dataset.
    Missing future variables are naturally handled as NaNs.
    """
    temporal_dicts = generate_temporal_datasets(X_base)
    X_list, y_list = [], []
    stages = {'admission': 0, 'day_1': 1, 'day_2': 2, 'day_3': 3}

    for stage_name, df_stage in temporal_dicts.items():
        df_stage = df_stage.copy()
        df_stage['TIMELINE_STAGE'] = stages[stage_name]
        X_list.append(df_stage)
        y_list.append(y_base.copy())

    X_aug = pd.concat(X_list, ignore_index=True)
    y_aug = pd.concat(y_list, ignore_index=True)
    return X_aug, y_aug

def fix_categorical_types(df, cat_columns):
    for col in cat_columns:
        if col in df.columns:
            df[col] = df[col].astype(str).replace({'nan': 'Unknown', 'NaN': 'Unknown'}).astype('category')
    df['TIMELINE_STAGE'] = df['TIMELINE_STAGE'].astype('category')
    return df
    
# Train/Test Split (Performed before augmentation to prevent leakage)
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)

print("Executing Temporal Augmentation on the training set...")
X_train_aug, y_train_aug = create_augmented_dataset(X_train_base, y_train_base)

Executing Temporal Augmentation on the training set...


# Approach 1: Multi-Task Neural Network (PyTorch)

**The Accuracy Paradox:** An initial unregularized approach resulted in class collapse. The neural network achieved >95% accuracy by predicting the majority class, but Recall was 0.00.

**The Regularized Solution:** 
We added `BatchNorm1d`, `Dropout`, and penalized weights (`pos_weight`) in the loss function to force the network to identify rare classes. 

*Results:* Recall increased significantly (e.g., 0.7250 for Heart Failure), but overall accuracy dropped due to false positives. Furthermore, temporal degradation was observed: medication data from subsequent days acted as statistical noise, worsening performance.

In [25]:
from sklearn.model_selection import train_test_split
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ---------------------------------------------------------
# 1. Data Processing and Validation Split
# ---------------------------------------------------------
cat_cols_nn = categorical_cols.copy() + ['TIMELINE_STAGE']
num_cols_nn = numeric_cols.copy()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, num_cols_nn),
        ('cat', categorical_transformer, cat_cols_nn)
    ])

X_train_processed = preprocessor.fit_transform(X_train_aug)
y_train_bin = y_train_aug[binary_targets].values.astype(np.float32)
y_train_multi = y_train_aug[multiclass_target].values.astype(np.int64)

X_t_nn, X_v_nn, y_b_t_nn, y_b_v_nn, y_m_t_nn, y_m_v_nn = train_test_split(
    X_train_processed, y_train_bin, y_train_multi, test_size=0.2, random_state=42
)

X_t_tensor = torch.tensor(X_t_nn, dtype=torch.float32)
y_b_t_tensor = torch.tensor(y_b_t_nn, dtype=torch.float32)
y_m_t_tensor = torch.tensor(y_m_t_nn, dtype=torch.long)

X_v_tensor = torch.tensor(X_v_nn, dtype=torch.float32)
y_b_v_tensor = torch.tensor(y_b_v_nn, dtype=torch.float32)
y_m_v_tensor = torch.tensor(y_m_v_nn, dtype=torch.long)

train_dataset = TensorDataset(X_t_tensor, y_b_t_tensor, y_m_t_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

val_dataset = TensorDataset(X_v_tensor, y_b_v_tensor, y_m_v_tensor)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

# ---------------------------------------------------------
# 2. Model Architecture
# ---------------------------------------------------------
class MultiTaskMI(nn.Module):
    def __init__(self, input_size, num_binary_targets, num_multiclass_classes, dropout_rate=0.3):
        super(MultiTaskMI, self).__init__()
        self.shared_fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)
        self.relu = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout_rate)
        self.shared_fc2 = nn.Linear(128, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.binary_head = nn.Linear(64, num_binary_targets)
        self.multiclass_head = nn.Linear(64, num_multiclass_classes)

    def forward(self, x):
        x = self.dropout1(self.relu(self.bn1(self.shared_fc1(x))))
        x = self.dropout2(self.relu(self.bn2(self.shared_fc2(x))))
        return self.binary_head(x), self.multiclass_head(x)

# ---------------------------------------------------------
# 3. Dynamic Penalty and Loss Functions
# ---------------------------------------------------------
num_positives = y_b_t_tensor.sum(dim=0)
num_negatives = y_b_t_tensor.shape[0] - num_positives
pos_weight_vector = (num_negatives / (num_positives + 1e-5)) * 50.0

criterion_binary = nn.BCEWithLogitsLoss(pos_weight=pos_weight_vector)
criterion_multiclass = nn.CrossEntropyLoss()

# ---------------------------------------------------------
# 4. Hyperparameter Tuning Loop
# ---------------------------------------------------------
learning_rates = [0.001, 0.005]
dropout_rates = [0.3, 0.5]

best_val_loss = float('inf')
best_params = {'lr': 0.001, 'dropout': 0.3}

print("Starting PyTorch Hyperparameter Tuning...")

for lr in learning_rates:
    for drop in dropout_rates:
        tune_model = MultiTaskMI(
            input_size=X_t_tensor.shape[1], 
            num_binary_targets=len(binary_targets), 
            num_multiclass_classes=8, 
            dropout_rate=drop
        )
        tune_optimizer = optim.Adam(tune_model.parameters(), lr=lr, weight_decay=1e-4)
        
        # Train for a few epochs to evaluate configuration
        for epoch in range(25):
            tune_model.train()
            for batch_X, batch_y_bin, batch_y_multi in train_loader:
                tune_optimizer.zero_grad()
                out_bin, out_multi = tune_model(batch_X)
                loss = criterion_binary(out_bin, batch_y_bin) + (criterion_multiclass(out_multi, batch_y_multi) * 0.5)
                loss.backward()
                tune_optimizer.step()
                
        # Validate configuration
        tune_model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y_bin, batch_y_multi in val_loader:
                out_bin, out_multi = tune_model(batch_X)
                loss = criterion_binary(out_bin, batch_y_bin) + (criterion_multiclass(out_multi, batch_y_multi) * 0.5)
                val_loss += loss.item()
                
        avg_val_loss = val_loss / len(val_loader)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_params['lr'] = lr
            best_params['dropout'] = drop

print(f"Optimal Parameters Identified: LR = {best_params['lr']}, Dropout = {best_params['dropout']}")

# ---------------------------------------------------------
# 5. Deep Training with Early Stopping
# ---------------------------------------------------------
print("\nStarting Deep Training with Optimal Parameters...")

model = MultiTaskMI(
    input_size=X_t_tensor.shape[1], 
    num_binary_targets=len(binary_targets), 
    num_multiclass_classes=8, 
    dropout_rate=best_params['dropout']
)
optimizer = optim.Adam(model.parameters(), lr=best_params['lr'], weight_decay=1e-4)

MAX_EPOCHS = 200
PATIENCE = 20
best_deep_val_loss = float('inf')
epochs_no_improve = 0
best_model_weights = None

for epoch in range(MAX_EPOCHS):
    model.train()
    train_loss = 0.0
    for batch_X, batch_y_bin, batch_y_multi in train_loader:
        optimizer.zero_grad()
        out_bin, out_multi = model(batch_X)
        loss = criterion_binary(out_bin, batch_y_bin) + (criterion_multiclass(out_multi, batch_y_multi) * 0.5)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation step
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_X, batch_y_bin, batch_y_multi in val_loader:
            out_bin, out_multi = model(batch_X)
            loss = criterion_binary(out_bin, batch_y_bin) + (criterion_multiclass(out_multi, batch_y_multi) * 0.5)
            val_loss += loss.item()
            
    avg_val_loss = val_loss / len(val_loader)
    
    # Track progress and early stopping
    if avg_val_loss < best_deep_val_loss:
        best_deep_val_loss = avg_val_loss
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1:03d}/{MAX_EPOCHS}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Early Stop Counter: {epochs_no_improve}/{PATIENCE}")
        
    if epochs_no_improve >= PATIENCE:
        print(f"\nEarly stopping triggered at epoch {epoch+1}. Restoring optimal weights.")
        break

# Load the best weights back into the final model
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)
    
print("PyTorch deep training completed.")

Starting PyTorch Hyperparameter Tuning...
Optimal Parameters Identified: LR = 0.005, Dropout = 0.3

Starting Deep Training with Optimal Parameters...
Epoch [010/200] | Train Loss: 2.8574 | Val Loss: 2.4042 | Early Stop Counter: 0/20
Epoch [020/200] | Train Loss: 2.6641 | Val Loss: 2.1113 | Early Stop Counter: 3/20
Epoch [030/200] | Train Loss: 2.4317 | Val Loss: 1.9325 | Early Stop Counter: 4/20
Epoch [040/200] | Train Loss: 2.0492 | Val Loss: 1.5914 | Early Stop Counter: 4/20
Epoch [050/200] | Train Loss: 2.0367 | Val Loss: 1.5794 | Early Stop Counter: 8/20
Epoch [060/200] | Train Loss: 2.4290 | Val Loss: 1.4768 | Early Stop Counter: 1/20
Epoch [070/200] | Train Loss: 1.8129 | Val Loss: 1.3902 | Early Stop Counter: 5/20
Epoch [080/200] | Train Loss: 1.8327 | Val Loss: 1.4202 | Early Stop Counter: 8/20
Epoch [090/200] | Train Loss: 1.6084 | Val Loss: 1.2268 | Early Stop Counter: 3/20
Epoch [100/200] | Train Loss: 1.8793 | Val Loss: 1.1500 | Early Stop Counter: 2/20
Epoch [110/200] | Tr

# Approach 2: Paradigm Shift with XGBoost

Deep Neural Networks struggle with high-dimensional tabular data and small sample sizes. Tree-based algorithms, such as XGBoost, outperform Deep Learning in these scenarios (Grinsztajn et al., 2022).

**Observed Advantages:**
* Handled NaNs generated by temporal masking natively.
* Improved data structuring and accuracy stability.
* However, it suffered from temporal stagnation: the admission data already contained the maximum predictive signal.

In [10]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, fbeta_score, confusion_matrix
from sklearn.utils.class_weight import compute_sample_weight
from ucimlrepo import fetch_ucirepo

# ---------------------------------------------------------
# 1. PREPARAÇÃO DOS DADOS E SPLITS
# ---------------------------------------------------------
print("="*70)
print("1. PREPARAÇÃO DOS DADOS E SPLITS")
print("="*70)

mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()

y_full = y_full.fillna(y_full.mode().iloc[0])

target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]

day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
drop_cols = day_1_cols + day_2_cols + day_3_cols

X_adm = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns]).copy()
X_adm = X_adm.apply(pd.to_numeric, errors='coerce')

X_train_base, X_test, y_train_base, y_test = train_test_split(
    X_adm, y_full, test_size=0.2, random_state=42
)

X_tune_train, X_tune_val, y_tune_train, y_tune_val = train_test_split(
    X_train_base, y_train_base, test_size=0.2, random_state=42
)

print(f"Base Train Volume (80%): {len(X_train_base)} | Final Test (20%): {len(X_test)}")
print(f"Tuning Split -> Train: {len(X_tune_train)} | Val: {len(X_tune_val)}")

# ---------------------------------------------------------
# 2. TUNING, THRESHOLDING E DEEP REFITTING
# ---------------------------------------------------------
print("\n" + "="*70)
print("2. TUNING SIMPLIFICADO E TREINAMENTO PROFUNDO (COM PESOS BALANCEADOS)")
print("="*70)

final_models = {}
MIN_RECALL_ACCEPTABLE = 0.70
thresholds_array = np.arange(0.001, 0.50, 0.001)

depths = [3, 5, 7]
learning_rates = [0.01, 0.05, 0.1]

for target in target_names:
    print(f"\n--- Processando Alvo: {target} ---")
    
    y_train_base_t = y_train_base[target].astype(int)
    y_test_t = y_test[target].astype(int)
    y_tune_train_t = y_tune_train[target].astype(int)
    y_tune_val_t = y_tune_val[target].astype(int)
    
    # Gera pesos balanceados para penalizar erros nas classes minoritárias
    weights_tune = compute_sample_weight('balanced', y_tune_train_t)
    weights_base = compute_sample_weight('balanced', y_train_base_t)
        
    best_val_score = -1.0
    best_params = {'max_depth': 5, 'learning_rate': 0.05}
    
    # Busca Grid Simplificada
    for d in depths:
        for lr in learning_rates:
            search_params = {
                'n_estimators': 500, 
                'max_depth': d,
                'learning_rate': lr,
                'subsample': 0.8,
                'colsample_bytree': 0.8,
                'tree_method': 'hist',
                'random_state': 42,
                'n_jobs': -1
            }
            
            model_search = xgb.XGBClassifier(**search_params)
            # Injeção direta dos pesos balanceados no treinamento
            model_search.fit(X_tune_train, y_tune_train_t, sample_weight=weights_tune)
            preds = model_search.predict(X_tune_val)
            
            score = fbeta_score(y_tune_val_t, preds, beta=2, zero_division=0) if target in binary_targets else accuracy_score(y_tune_val_t, preds)
            
            if score > best_val_score:
                best_val_score = score
                best_params['max_depth'] = d
                best_params['learning_rate'] = lr

    print(f"Parâmetros Ótimos: Profundidade={best_params['max_depth']}, LR={best_params['learning_rate']}")

    # Treinamento longo com paciência estendida para evitar paradas prematuras
    tune_params = {
        'n_estimators': 2000, 
        'max_depth': best_params['max_depth'],
        'learning_rate': best_params['learning_rate'],
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'tree_method': 'hist',
        'random_state': 42,
        'n_jobs': -1,
        'early_stopping_rounds': 500, # Paciência alta para forçar a busca
        'eval_metric': 'aucpr' if target in binary_targets else 'mlogloss'
    }
    
    model_tune = xgb.XGBClassifier(**tune_params)
    model_tune.fit(
        X_tune_train, y_tune_train_t,
        sample_weight=weights_tune,
        eval_set=[(X_tune_val, y_tune_val_t)],
        verbose=False
    )
    
    optimal_trees = model_tune.best_iteration
    print(f"Número ideal de árvores consolidado: {optimal_trees}")
    
    # Determinação de Limiar
    final_threshold = 0.50
    if target in binary_targets:
        y_val_probs = model_tune.predict_proba(X_tune_val)[:, 1]
        best_acc_at_recall = 0.0
        best_f2 = 0.0
        fallback_thresh = 0.50
        
        for thresh in thresholds_array:
            y_val_pred = (y_val_probs >= thresh).astype(int)
            rec = recall_score(y_tune_val_t, y_val_pred, zero_division=0)
            acc = accuracy_score(y_tune_val_t, y_val_pred)
            f2 = fbeta_score(y_tune_val_t, y_val_pred, beta=2, zero_division=0)
            
            if f2 > best_f2:
                best_f2 = f2
                fallback_thresh = thresh
                
            if rec >= MIN_RECALL_ACCEPTABLE and acc > best_acc_at_recall:
                best_acc_at_recall = acc
                final_threshold = thresh
                
        if best_acc_at_recall == 0:
            final_threshold = fallback_thresh
            
        print(f"Limiar Científico Definido: {final_threshold:.3f}")

    # Retreinamento na base completa de 80%
    refit_params = tune_params.copy()
    refit_params['n_estimators'] = optimal_trees
    refit_params.pop('early_stopping_rounds', None)
    refit_params.pop('eval_metric', None)
    
    model_final = xgb.XGBClassifier(**refit_params)
    model_final.fit(X_train_base, y_train_base_t, sample_weight=weights_base)
    
    final_models[target] = (model_final, final_threshold)

# ---------------------------------------------------------
# 3. AVALIAÇÃO FINAL NO CONJUNTO DE TESTE (20%)
# ---------------------------------------------------------
print("\n" + "="*70)
print("3. AVALIAÇÃO FINAL NA BASE DE TESTE INÉDITA")
print("="*70)

for target in target_names:
    print(f"\n--- Resultados Finais: {target} ---")
    model_final, threshold = final_models[target]
    y_test_t = y_test[target].astype(int).values
    
    if target in binary_targets:
        y_test_probs = model_final.predict_proba(X_test)[:, 1]
        y_test_pred = (y_test_probs >= threshold).astype(int)
        
        test_rec = recall_score(y_test_t, y_test_pred, zero_division=0)
        test_acc = accuracy_score(y_test_t, y_test_pred)
        test_cm = confusion_matrix(y_test_t, y_test_pred)
        
        print(f"Recall (Teste): {test_rec:.4f} | Acurácia (Teste): {test_acc:.4f}")
        print("Matriz de Confusão:")
        print(test_cm)
    else:
        y_test_pred = model_final.predict(X_test)
        
        test_acc = accuracy_score(y_test_t, y_test_pred)
        test_rec = recall_score(y_test_t, y_test_pred, average='weighted', zero_division=0)
        test_cm = confusion_matrix(y_test_t, y_test_pred)
        
        print(f"Acurácia: {test_acc:.4f} | Weighted Recall: {test_rec:.4f}")
        print("Matriz de Confusão:")
        print(test_cm)

1. PREPARAÇÃO DOS DADOS E SPLITS
Base Train Volume (80%): 1360 | Final Test (20%): 340
Tuning Split -> Train: 1088 | Val: 272

2. TUNING SIMPLIFICADO E TREINAMENTO PROFUNDO (COM PESOS BALANCEADOS)

--- Processando Alvo: FIBR_PREDS ---
Parâmetros Ótimos: Profundidade=3, LR=0.01
Número ideal de árvores consolidado: 1988
Limiar Científico Definido: 0.112

--- Processando Alvo: PREDS_TAH ---
Parâmetros Ótimos: Profundidade=3, LR=0.01
Número ideal de árvores consolidado: 500
Limiar Científico Definido: 0.080

--- Processando Alvo: JELUD_TAH ---
Parâmetros Ótimos: Profundidade=3, LR=0.01
Número ideal de árvores consolidado: 18
Limiar Científico Definido: 0.430

--- Processando Alvo: FIBR_JELUD ---
Parâmetros Ótimos: Profundidade=3, LR=0.01
Número ideal de árvores consolidado: 15
Limiar Científico Definido: 0.485

--- Processando Alvo: A_V_BLOK ---
Parâmetros Ótimos: Profundidade=7, LR=0.01
Número ideal de árvores consolidado: 0
Limiar Científico Definido: 0.496

--- Processando Alvo: OTEK_LA

# Approach 3: AutoGluon, Classifier Chains, and Threshold Optimization

To bypass the flaws of independent approaches, we adopted AutoGluon adapted for the clinical "small data" scope:

1. **Banning Deep Learning:** Preventing overfitting.
2. **Classifier Chains:** Modeling comorbidities. The prediction for Target 1 becomes an input for Target 2, respecting the biology of the failure cascade.
3. **Threshold Sweep:** Adjusting the decision threshold mathematically (abandoning the 50% default) to guarantee a minimum Recall of 70% in risk triage.

In [12]:
import os
from autogluon.tabular import TabularPredictor
import numpy as np

print("\n" + "="*70)
print("TRAINING & LOADING CLASSIFIER CHAIN WITH EXPLICIT PATHS")
print("="*70)

# Define a base directory for all chain models
BASE_MODELS_DIR = "AutogluonModels/ClinicalChain"

X_adm = generate_temporal_datasets(X_full)['admission']
df_full_adm = pd.concat([X_adm, y_full], axis=1)

train_data, test_data = train_test_split(df_full_adm, test_size=0.2, random_state=42)
train_data = TabularDataset(train_data)
test_data = TabularDataset(test_data)
predictors = {}
predicted_train_features = train_data.copy()
predicted_test_features = test_data.copy()

for i, target in enumerate(target_names):
    print(f"\n--- Processing Target: {target} ({i+1}/{len(target_names)}) ---")
    
    # Define the explicit path for this specific target
    model_path = os.path.join(BASE_MODELS_DIR, target)
    
    # 1. Prepare data for the current step in the chain
    future_targets = target_names[i+1:]
    train_input = predicted_train_features.drop(columns=future_targets).copy()
    test_input = predicted_test_features.drop(columns=future_targets).copy()

    # 2. Check if the model is already trained and saved on disk
    if os.path.exists(model_path):
        print(f"Model for {target} found on disk. Loading from {model_path}...")
        predictor = TabularPredictor.load(model_path)
    else:
        print(f"Model for {target} not found. Starting training...")
        
        # Apply sample weights for binary targets to handle extreme imbalance
        if target in binary_targets:
            num_pos = (train_input[target] == 1).sum()
            num_neg = (train_input[target] == 0).sum()
            weight_ratio = num_neg / (num_pos + 1e-5)
            train_input['sample_weight'] = np.where(train_input[target] == 1, weight_ratio, 1.0)
            weight_col = 'sample_weight'
            prob_type = 'binary'
            metric = 'roc_auc'
        else:
            weight_col = None
            prob_type = 'multiclass'
            metric = 'accuracy'

        # Calling the AutoGulon!
        predictor = TabularPredictor(
            label=target, 
            eval_metric=metric,
            problem_type=prob_type,
            sample_weight=weight_col,
            path=model_path
        ).fit(
            train_input, 
            presets='best_quality',
            time_limit=60,
            excluded_model_types=['FASTAI'], 
            verbosity=0 
        )

    # Store the predictor in the session dictionary
    predictors[target] = predictor

    # 3. Predict and append the result to the features for the next chain link
    if 'sample_weight' in train_input.columns:
        train_input = train_input.drop(columns=['sample_weight'])

    predicted_train_features[target] = predictor.predict(train_input)
    predicted_test_features[target] = predictor.predict(test_input)

print("\nChain processing complete. All models are available in the predictors dictionary.")


TRAINING & LOADING CLASSIFIER CHAIN WITH EXPLICIT PATHS

--- Processing Target: FIBR_PREDS (1/12) ---
Model for FIBR_PREDS found on disk. Loading from AutogluonModels/ClinicalChain/FIBR_PREDS...

--- Processing Target: PREDS_TAH (2/12) ---
Model for PREDS_TAH found on disk. Loading from AutogluonModels/ClinicalChain/PREDS_TAH...

--- Processing Target: JELUD_TAH (3/12) ---
Model for JELUD_TAH found on disk. Loading from AutogluonModels/ClinicalChain/JELUD_TAH...

--- Processing Target: FIBR_JELUD (4/12) ---
Model for FIBR_JELUD found on disk. Loading from AutogluonModels/ClinicalChain/FIBR_JELUD...

--- Processing Target: A_V_BLOK (5/12) ---
Model for A_V_BLOK not found. Starting training...

--- Processing Target: OTEK_LANC (6/12) ---
Model for OTEK_LANC not found. Starting training...

--- Processing Target: RAZRIV (7/12) ---
Model for RAZRIV not found. Starting training...

--- Processing Target: DRESSLER (8/12) ---
Model for DRESSLER not found. Starting training...

--- Processing

# 4. Model Comparison and Threshold Optimization

**Methodology for Clinical Evaluation**

To properly evaluate the PyTorch, XGBoost, and AutoGluon models, standard accuracy is insufficient due to the extreme class imbalance (Accuracy Paradox). We employ a custom evaluation strategy focused on high-sensitivity triage:

1. **Log Loss (Cross-Entropy):** Evaluates the raw probabilistic confidence of each model before any hard threshold is applied.
2. **Threshold Sweeping:** Clinical alerts cannot rely on the default 0.5 probability threshold. The algorithm iteratively tests thresholds between 0.01 and 0.99, seeking to maximize global Accuracy while strictly maintaining a minimum Recall of 80% (0.80).
3. **F3-Score ($F_\beta$ with $\beta=3$):** Weighs Recall 9 times higher than Precision. This mathematical asymmetry penalizes False Negatives heavily, aligning with the medical imperative of missing zero critical complications.

In [27]:
from sklearn.metrics import fbeta_score, log_loss, accuracy_score, recall_score, confusion_matrix
import numpy as np
import pandas as pd
import torch

print("\n" + "="*70)
print("MODEL COMPARISON & THRESHOLD OPTIMIZATION (ADMISSION DATASET)")
print("="*70)

MIN_RECALL_ACCEPTABLE = 0.666
thresholds_to_test = np.arange(0.01, 0.99, 0.01)
comparison_results = []

def optimize_threshold_and_evaluate(y_true, y_probs, model_name, target_name):
    best_threshold = 0.50
    best_recall = 0.0
    best_acc = 0.0
    
    # Sweep thresholds to find highest accuracy that meets the recall constraint
    for thresh in thresholds_to_test:
        y_pred_thresh = (y_probs >= thresh).astype(int)
        rec = recall_score(y_true, y_pred_thresh, zero_division=0)
        acc = accuracy_score(y_true, y_pred_thresh)
        
        if rec >= MIN_RECALL_ACCEPTABLE and acc > best_acc:
            best_acc = acc
            best_threshold = thresh
            best_recall = rec
            
    # Fallback: if no threshold achieves the minimum recall, default to the lowest threshold
    # to maximize recall at the expense of accuracy.
    if best_acc == 0.0:
        best_threshold = thresholds_to_test[0]
        y_pred_fallback = (y_probs >= best_threshold).astype(int)
        best_acc = accuracy_score(y_true, y_pred_fallback)
        best_recall = recall_score(y_true, y_pred_fallback, zero_division=0)

    # Final predictions with the optimal threshold
    y_pred_final = (y_probs >= best_threshold).astype(int)
    
    # Calculate final metrics
    f3 = fbeta_score(y_true, y_pred_final, beta=3, zero_division=0)
    loss = log_loss(y_true, y_probs)
    cm = confusion_matrix(y_true, y_pred_final)
    
    return {
        'Model': model_name,
        'Target': target_name,
        'Opt_Threshold': best_threshold,
        'Accuracy': best_acc,
        'Recall': best_recall,
        'F3_Score': f3,
        'Log_Loss': loss
    }

# ---------------------------------------------------------
# 1. PyTorch Evaluation (Admission)
# ---------------------------------------------------------
# Assuming model, preprocessor, and dataloaders are already defined in the environment
X_test_adm_pt = generate_temporal_datasets(X_test_base)['admission']
X_test_adm_pt = X_test_adm_pt.copy()
X_test_adm_pt['TIMELINE_STAGE'] = 0 # 0 maps to 'admission'

expected_cols = preprocessor.feature_names_in_.tolist()
for col in expected_cols:
    if col not in X_test_adm_pt.columns:
        X_test_adm_pt[col] = np.nan
        
# Re-add missing columns to match preprocessing expectations
for col in X_train_base.columns:
    if col not in X_test_adm_pt.columns:
        X_test_adm_pt[col] = np.nan

X_test_adm_pt = X_test_adm_pt[expected_cols]
X_test_processed_pt = preprocessor.transform(X_test_adm_pt)
X_test_tensor_eval = torch.tensor(X_test_processed_pt, dtype=torch.float32)

model.eval()
with torch.no_grad():
    test_out_bin, _ = model(X_test_tensor_eval)
    pt_probs = torch.sigmoid(test_out_bin).numpy()

# Evaluate PyTorch binary targets (Using the first 2 targets as an example scope)
for i, target in enumerate(binary_targets): 
    y_true = y_test_base[target].values.astype(int)
    y_probs = pt_probs[:, i]
    res = optimize_threshold_and_evaluate(y_true, y_probs, 'PyTorch_NN', target)
    comparison_results.append(res)

# ---------------------------------------------------------
# 2. XGBoost Evaluation (Admission)
# ---------------------------------------------------------
X_test_xgb = X_test.copy()
for col in X_train_base.columns:
    if col not in X_test_xgb.columns:
        X_test_xgb[col] = np.nan

X_test_xgb = X_test_xgb[X_train_base.columns]

for target in binary_targets:
    y_true = y_test[target].values.astype(int)
    xgb_model_final, _ = final_models[target]
    y_probs = xgb_model_final.predict_proba(X_test_xgb)[:, 1]
    res = optimize_threshold_and_evaluate(y_true, y_probs, 'XGBoost_Optimized', target)
    comparison_results.append(res)
    
# ---------------------------------------------------------
# 3. AutoGluon Classifier Chain Evaluation (Admission)
# ---------------------------------------------------------
chain_test_features = generate_temporal_datasets(X_test_base)['admission'].copy()

for i, target in enumerate(target_names):
    y_true = y_test_base[target].values.astype(int)
    
    if target in binary_targets:
        y_probs = predictors[target].predict_proba(chain_test_features).iloc[:, 1].values
        res = optimize_threshold_and_evaluate(y_true, y_probs, 'AutoGluon_Chain', target)
        comparison_results.append(res)
        
    y_preds_for_chain = predictors[target].predict(chain_test_features)
    chain_test_features[target] = y_preds_for_chain

# ---------------------------------------------------------
# 4. Final Comparison DataFrame
# ---------------------------------------------------------
df_comparison = pd.DataFrame(comparison_results)
df_comparison = df_comparison.sort_values(by=['Target', 'F3_Score'], ascending=[True, False]).reset_index(drop=True)

print("\nFINAL METRICS (Sorted by Target and F3-Score):")
print(df_comparison.to_string())


MODEL COMPARISON & THRESHOLD OPTIMIZATION (ADMISSION DATASET)

FINAL METRICS (Sorted by Target and F3-Score):
                Model      Target  Opt_Threshold  Accuracy    Recall  F3_Score  Log_Loss
0     AutoGluon_Chain    A_V_BLOK           0.17  0.929412  0.727273  0.625000  0.173412
1          PyTorch_NN    A_V_BLOK           0.01  0.570588  0.636364  0.282258  0.515083
2   XGBoost_Optimized    A_V_BLOK           0.01  0.032353  1.000000  0.250569  0.957901
3     AutoGluon_Chain    DRESSLER           0.19  0.582353  0.705882  0.397351  0.280274
4   XGBoost_Optimized    DRESSLER           0.48  0.397059  0.764706  0.354223  0.668127
5          PyTorch_NN    DRESSLER           0.01  0.535294  0.588235  0.318471  1.044371
6     AutoGluon_Chain  FIBR_JELUD           0.42  0.717647  0.692308  0.412844  0.453544
7   XGBoost_Optimized  FIBR_JELUD           0.46  0.529412  0.846154  0.384615  0.643458
8          PyTorch_NN  FIBR_JELUD           0.01  0.576471  0.538462  0.267176  0.852276

# 5 -> Approach 4: Using TabFPN 2.5

This architeture could be a new way to improve the overall model quality

In [ ]:
!pip install "tabpfn @ git+https://github.com/PriorLabs/TabPFN.git"

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

hf_token = os.getenv("HF_TOKEN")

os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, accuracy_score, fbeta_score, confusion_matrix
from ucimlrepo import fetch_ucirepo

# Importação do Modelo Fundacional (baixará os pesos na primeira execução)
from tabpfn import TabPFNClassifier

print("="*70)
print("1. PREPARAÇÃO DOS DADOS E SPLITS")
print("="*70)

mi_data = fetch_ucirepo(id=579)
X_full = mi_data.data.features.copy()
y_full = mi_data.data.targets.copy()

y_full = y_full.fillna(y_full.mode().iloc[0])

target_names = y_full.columns.tolist()
binary_targets = target_names[:-1]

# Mascaramento Temporal (Admissão)
day_1_cols = ['R_AB_1_n', 'NA_R_1_n', 'NOT_NA_1_n']
day_2_cols = ['R_AB_2_n', 'NA_R_2_n', 'NOT_NA_2_n']
day_3_cols = ['R_AB_3_n', 'NA_R_3_n', 'NOT_NA_3_n']
drop_cols = day_1_cols + day_2_cols + day_3_cols

X_adm = X_full.drop(columns=[c for c in drop_cols if c in X_full.columns]).copy()
X_adm = X_adm.apply(pd.to_numeric, errors='coerce')

# Split Triplo
X_train_base, X_test, y_train_base, y_test = train_test_split(
    X_adm, y_full, test_size=0.2, random_state=42
)

X_tune_train, X_tune_val, y_tune_train, y_tune_val = train_test_split(
    X_train_base, y_train_base, test_size=0.2, random_state=42
)

print(f"Volume Treino (Contexto): {len(X_tune_train)} | Validação: {len(X_tune_val)} | Teste Final: {len(X_test)}")

print("\n" + "="*70)
print("2. INFERÊNCIA ZERO-SHOT (TABPFN-2.6) E AVALIAÇÃO DE LIMIAR")
print("="*70)

tabpfn_models = {}
MIN_RECALL_ACCEPTABLE = 0.70
thresholds = np.arange(0.05, 0.95, 0.01)

for target in target_names:
    print(f"\n--- Processando Alvo: {target} ---")
    
    y_train_t = y_tune_train[target].astype(int)
    y_val_t = y_tune_val[target].astype(int).values
    y_test_t = y_test[target].astype(int).values
    
    # Instancia o TabPFN (Sem parâmetros complexos de busca)
    model_tabpfn = TabPFNClassifier(random_state=42, ignore_pretraining_limits=True)
    
    # Carrega a base de treino no contexto do modelo
    model_tabpfn.fit(X_tune_train, y_train_t)
    tabpfn_models[target] = model_tabpfn
    
    if target in binary_targets:
        # A. DESCOBERTA DE LIMIAR NA VALIDAÇÃO
        # O modelo processa os dados de validação baseados no contexto do treino
        y_val_probs = model_tabpfn.predict_proba(X_tune_val)[:, 1]
        
        best_thresh = 0.50
        best_acc_at_recall = 0.0
        best_f2 = 0.0
        fallback_thresh = 0.50
        
        for thresh in thresholds:
            y_val_pred = (y_val_probs >= thresh).astype(int)
            rec = recall_score(y_val_t, y_val_pred, zero_division=0)
            acc = accuracy_score(y_val_t, y_val_pred)
            f2 = fbeta_score(y_val_t, y_val_pred, beta=2, zero_division=0)
            
            if f2 > best_f2:
                best_f2 = f2
                fallback_thresh = thresh
                
            if rec >= MIN_RECALL_ACCEPTABLE and acc > best_acc_at_recall:
                best_acc_at_recall = acc
                best_thresh = thresh
                
        final_threshold = best_thresh if best_acc_at_recall > 0 else fallback_thresh
        
        # B. APLICAÇÃO NA BASE DE TESTE INÉDITA
        y_test_probs = model_tabpfn.predict_proba(X_test)[:, 1]
        y_test_pred = (y_test_probs >= final_threshold).astype(int)
        
        test_rec = recall_score(y_test_t, y_test_pred, zero_division=0)
        test_acc = accuracy_score(y_test_t, y_test_pred)
        test_cm = confusion_matrix(y_test_t, y_test_pred)
        
        print(f"Limiar Científico Definido: {final_threshold:.2f}")
        print(f"Recall (Teste): {test_rec:.4f} | Acurácia (Teste): {test_acc:.4f}")
        print("Matriz de Confusão:")
        print(test_cm)
        
    else:
        # Avaliação do alvo multiclasse (LET_IS)
        y_test_pred = model_tabpfn.predict(X_test)
        test_acc = accuracy_score(y_test_t, y_test_pred)
        test_rec = recall_score(y_test_t, y_test_pred, average='weighted', zero_division=0)
        test_cm = confusion_matrix(y_test_t, y_test_pred)
        
        print(f"Acurácia: {test_acc:.4f} | Weighted Recall: {test_rec:.4f}")
        print("Matriz de Confusão:")
        print(test_cm)

In [ ]:
import numpy as np
from sklearn.metrics import recall_score, accuracy_score, fbeta_score, confusion_matrix

print("\n" + "="*70)
print("VARREDURA DE LIMIAR DE ALTA PRECISÃO (TABPFN)")
print("="*70)

# Meta clínica: 70% de captura dos doentes
MIN_RECALL_ACCEPTABLE = 0.70

# Array de 0.001 até 0.900 com passos de 0.001 (900 testes por alvo)
thresholds = np.arange(0.001, 0.15, 0.001)

for target in binary_targets:
    print(f"\n--- Otimização Fina para o Alvo: {target} ---")
    
    y_val_t = y_tune_val[target].astype(int).values
    y_test_t = y_test[target].astype(int).values
    
    # 1. Extração das probabilidades contínuas na Validação
    y_val_probs = tabpfn_models[target].predict_proba(X_tune_val)[:, 1]
    
    best_thresh = 0.500
    best_acc_at_recall = 0.0
    best_f2 = 0.0
    fallback_thresh = 0.500
    
    # 2. Varredura dos 900 limiares
    for thresh in thresholds:
        y_val_pred = (y_val_probs >= thresh).astype(int)
        
        rec = recall_score(y_val_t, y_val_pred, zero_division=0)
        acc = accuracy_score(y_val_t, y_val_pred)
        f2 = fbeta_score(y_val_t, y_val_pred, beta=2, zero_division=0)
        
        # Atualiza o plano de contingência (Maior F2-Score global)
        if f2 > best_f2:
            best_f2 = f2
            fallback_thresh = thresh
            
        # Atualiza a regra de negócio (Maior acurácia com Recall >= 70%)
        if rec >= MIN_RECALL_ACCEPTABLE and acc > best_acc_at_recall:
            best_acc_at_recall = acc
            best_thresh = thresh
            
    # Define o limiar matemático estrito
    final_threshold = best_thresh if best_acc_at_recall > 0 else fallback_thresh
    
    # 3. Avaliação cega na base de Teste Inédita
    y_test_probs = tabpfn_models[target].predict_proba(X_test)[:, 1]
    y_test_pred = (y_test_probs >= final_threshold).astype(int)
    
    test_rec = recall_score(y_test_t, y_test_pred, zero_division=0)
    test_acc = accuracy_score(y_test_t, y_test_pred)
    test_cm = confusion_matrix(y_test_t, y_test_pred)
    
    # 4. Exibição formatada
    print(f"Limiar Ótimo Encontrado: {final_threshold:.3f}")
    print(f"Recall (Teste): {test_rec:.4f} | Acurácia (Teste): {test_acc:.4f}")
    print("Matriz de Confusão:")
    print(test_cm)